## Phikon simple_triton example

This notebook illustrates how to encode tiles from a whole-slide image using a Triton inference server python backend for phikon and can be modified for UNI model too.

Notes for running phikon (huggingface) using python backend on triton server container.
 - In the triton server container install requirements.txt 
 - The phikon folder contains model.py that should be inside folder `1` and config.pbtxt should be in the same level of folder 1.
    The phikon folder should be manually moved to the `host_model_repository` folder. 

        .
        ├── ...
        ├── phikon                  # Phikon Model
        │   ├── config.pbtxt        # Default configuration file
        │   ├── 1                   # Version 1 of the model
        │       ├── model.py        # TritonPythonModel Class adapted for Phikon

Notes for running:
- See https://github.com/PathologyDataScience/simple_triton for details on launching the triton server container and mounting the model repository notebook
- This notebook requires installation of `mil`, `glimr`, and `histomics_stream`
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount your model repository directory to the triton container

In [ ]:
# install large_image with tile sources
!pip install histomics_stream 'large_image[tiff]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install ../../simple_triton
    
# install ray tune dependencies and mil
!pip install pooch
!pip install pyarrow
!pip install ray
!pip install ../../glimr
!pip install ../../mil

## Run client with no GPUs

If running Triton and the client on the same machine, we want to stop the client tensorflow from consuming GPU resources. By default, TensorFlow maps nearly all available GPU memory.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf

assert len(tf.config.list_physical_devices("GPU")) == 0

2024-05-01 17:06:12.797678: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-05-01 17:06:12.860055: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-01 17:06:13.724939: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-05-01 17:06:14.818469: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:282] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2024-05-01 17:06:14.818534: I external/local_xla/xla/stream_executor/cuda/cu

## Download sample data

Download the hosted whole-slide image and mask.

In [2]:
import pooch

# download whole slide image and corresponding mask
wsi_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.svs",
    url="https://drive.google.com/uc?export=download&id=19agE_0cWY582szhOVxp9h3kozRfB4CvV&confirm=t&uuid=6f2d51e7-9366-4e98-abc7-4f77427dd02c&at=ALgDtswlqJJw1KU7P3Z1tZNcE01I:1679111148632",
    known_hash="d046f952759ff6987374786768fc588740eef1e54e4e295a684f3bd356c8528f",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)
mask_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.mask.png",
    url="https://drive.google.com/uc?export=download&id=17GOOHbL8Bo3933rdIui82akr7stbRfta",
    known_hash="bb657ead9fd3b8284db6ecc1ca8a1efa57a0e9fd73d2ea63ce6053fbd3d65171",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)

## Load the model using `TritonModel`

After creating the model, we load the model into Triton using the `TritonModel` class. This class contains methods for loading, unloading, and checking the status of models. To load the model we read the configuration with batch size 128. By default it will load a single copy of the model on each system GPU, and will often automatically set optimizations like pinned memory.

In [3]:
from simple_triton.model import TritonModel
from pprint import pprint
url = "localhost:8001"  # url for grpc access to tirton server
model_name = "phikon"  # set model name "uni"

#Load the model. Currently can't be loaded with a dynamic configuration. It will load the config.pbtxt
model = TritonModel(model_name, url)
model.unload()
model.load()
assert model.is_loaded()
pprint(model.get_config())

{'backend': 'python',
 'defaultModelFilename': 'model.py',
 'input': [{'dataType': 'TYPE_UINT8',
            'dims': ['224', '224', '3'],
            'name': 'input_0'}],
 'instanceGroup': [{'count': 1,
                    'gpus': [0, 1, 2, 3, 4, 5, 6],
                    'kind': 'KIND_GPU',
                    'name': 'phikon_0'}],
 'maxBatchSize': 128,
 'name': 'phikon',
 'optimization': {'inputPinnedMemory': {'enable': True},
                  'outputPinnedMemory': {'enable': True}},
 'output': [{'dataType': 'TYPE_FP32', 'dims': ['768'], 'name': 'output_0'}],
 'versionPolicy': {'latest': {'numVersions': 1}}}


## Setup Configuration with `ConfigBuilder`

The `ConfigBuilder` class provides access to advanced configuration options like backend optimizations. Here, we create a duplicate model on each GPU (`count=2`) and convert the model to mixed precision to improve speed and memory usage. The model is reloaded using this advanced configuration.

In [4]:
from simple_triton.config import ConfigBuilder

batch = 128
count = 2 # set gpu count
kind = "gpu"  # set gpu kind
gpus = 7  # set number of gpus

#Set max_batch size 
config = model.get_config()
config["max_batch_size"] = batch
config_builder = ConfigBuilder(model_name=model_name, config=config, url=url)


# increase the number of model instances per GPU to 2
# The configuration is written to include 7 gpus. However, if you need to change that you need to pass the gpus as a list. 
if kind == "gpu":
    start, end, intval = 0, gpus, 1
    gpus = list(range(start, end, intval))
    config_builder.remove_instance_groups()
    config_builder.add_instance_group(count, kind, gpus)

model.load(config=config_builder.config)

assert model.is_loaded()
pprint(model.get_config())

{'backend': 'python',
 'defaultModelFilename': 'model.py',
 'input': [{'dataType': 'TYPE_UINT8',
            'dims': ['224', '224', '3'],
            'name': 'input_0'}],
 'instanceGroup': [{'count': 2,
                    'gpus': [0, 1, 2, 3, 4, 5, 6],
                    'kind': 'KIND_GPU',
                    'name': 'phikon_0'}],
 'maxBatchSize': 128,
 'name': 'phikon',
 'optimization': {'inputPinnedMemory': {'enable': True},
                  'outputPinnedMemory': {'enable': True}},
 'output': [{'dataType': 'TYPE_FP32', 'dims': ['768'], 'name': 'output_0'}],
 'versionPolicy': {'latest': {'numVersions': 1}}}


## Run the inference

First, a histomics stream study is created defining the tiles that need to be read based on the whole-slide image, tissue mask, and desired magnification, tile size, and tile overlap. The chunk parameter is used to group tiles during disk reads to maximize throughput. This study initializes a `LargeimagePrefetch` iterator that generates batches of tiles and tile metadata using prefetching.

This iterator is passed to the inference function that is parameterized by the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

Loading is done with a np.uint8 datatype and that is how the configuration file is setup. 

In [5]:
from simple_triton.feature_extraction import inference, study
from simple_triton.tile_iterators import TiffPrefetch
from simple_triton.utils import analyze
from time import time
import numpy as np

# slide parameters
batch = 128
magnification = 20.0
chunk = 896
mask_threshold = 0.5
tile = 224

# create a histomics-stream study from a wsi/mask pair
hs_study = study(
    (wsi_path, mask_path),
    t=(tile, tile),
    chunk=(chunk, chunk),
    objective=magnification,
    mask_threshold=mask_threshold,
)

# tile iterator parameters
batch = 128
prefetch = 4
workers = 16  # total number of tile
icc = True  # apply ICC color correction

# inference parameters
limit = 1  # limit on number of pending requests per worker
verbose = True  # display inference statistics and debugging information

# start timer
start = time()

# create tile iterator
iterator = TiffPrefetch(hs_study, np.uint8, icc, batch, prefetch, workers)

# inference
features, metadata, times, failures = inference(
    iterator, model_name, url="localhost:8001", limit=limit, rest=0.0
)

# display elapsed time
print(f"Total elapsed time: {time()-start}")

# analyze performance
analyze(times)

Total elapsed time: 34.54744482040405
                        median    stdv    min    max
--------------------  --------  ------  -----  -----
total (sec)               0.86    0.14   0.34   1.20
read (sec)                0.01    0.00   0.00   0.02
submission (sec)          0.02    0.00   0.01   0.03
completion (sec)          0.84    0.14   0.33   1.18
retrieval (sec)           0.00    0.00   0.00   0.00
--------------------  --------  ------  -----  -----
read (% total)            0.76    0.33   0.20   1.45
submission (% total)      2.04    0.41   1.28   3.20
completion (% total)     96.90    0.62  95.50  98.38
retrieval (% total)       0.19    0.07   0.05   0.50


# Write features to .tfr

In [ ]:
from mil.io.reader import read_record, peek
from mil.io.writer import write_record

# concatenate features
features = np.concatenate(features[0], axis=0)

# create dummy labels
labels = {"labels": np.random.uniform(size=(10))}

# write to tfrecord
write_record(
    "./triton.tfr", features, metadata, labels, structured=False, precision=tf.float16
)

# get list of .tfr variables for de-serialization
serialized = list(tf.data.TFRecordDataset(["./triton.tfr"]))[0]
variables = peek(serialized)

# verify reading
read_record(serialized, variables, structured=False, precision=tf.float16)